# 补丁：`sales_relative_main` → `production_relative_main`

一次性脚本。**02 重跑一遍就不需要这个了**——写它只是为了避免在内存紧张时重跑整条大数据链。

## 要改什么

| 旧列 | 新列 | 定义 |
|---|---|---|
| `main_product_output`（主产品 `total_output`） | `main_product_production` | 主产品的 `production_value` |
| `sales_relative_main` = 副 sales / 主 sales | `production_relative_main` | 副 `production_value` / 主 `production_value` |

旧口径下分母是该 firm-year 最大的销量，比值恒 ≤ 1；主产品改按 `production_value` 选之后分母跟最大值脱钩，转售型企业会爆（实测均值 0.20 → 180 万）。分子分母同用 `production_value` 才重新有界。

## 为什么不用重跑上游

新列完全可以从现有 `full_data.dta` 自己算出来：**主产品行（`is_main == 1`）的 `production_value` 就是 `main_product_production`**。主产品身份（`is_main` / `main_product`）在 02 里已经是按 `production_value` 选的，不需要动。

## 两步，可以只跑第一步

- **Step 1**（很轻）：生成 `main_product_production.dta` 查找表，12.3M 行 × 3 列。
- **Step 2**（重，需要能装下整个 90M × 30）：把 `full_data.dta` 整体改写。

**04 / 05 的回归都用不到这两列**（Block 1 完全不用；Block 2 只在从未进论文的 "Sales" 分支里用），所以内存紧张时**可以只跑 Step 1，先往下做回归**。

In [ ]:
import pandas as pd
import numpy as np
import gc
from pathlib import Path

pd.set_option('display.width', 200)

DATA = Path(r'G:\Kuangyu_Temp\Outsource\Empirical1_data')
# DATA = Path(r'C:\Users\HKUBS\Documents\aproject\Outsourcing\Empirical1_data')   # 本地

FULL   = DATA / 'full_data.dta'
LOOKUP = DATA / 'main_product_production.dta'

print(FULL, '->', f'{FULL.stat().st_size/1e9:.2f} GB')

## Step 1　生成 `main_product_production` 查找表

分块扫，只取 `is_main == 1` 的行。每个 firm-year 恰好一行，12.3M 行。

In [ ]:
parts = []
for i, ch in enumerate(pd.read_stata(FULL, columns=['year', 'firm_id', 'is_main',
                                                     'production_value'],
                                     chunksize=10_000_000), 1):
    parts.append(ch.loc[ch['is_main'] == 1, ['year', 'firm_id', 'production_value']])
    print(f'  chunk {i}')

mainprod = (pd.concat(parts, ignore_index=True)
            .rename(columns={'production_value': 'main_product_production'}))
del parts; gc.collect()

print('firm-year 数:', f'{len(mainprod):,}')
print('重复检查:', mainprod.duplicated(subset=['year', 'firm_id']).sum(), '（应为 0）')
print(mainprod['main_product_production'].describe().to_string())

mainprod.to_stata(LOOKUP, write_index=False)
print('已保存:', LOOKUP.name)

## Step 2　改写 `full_data.dta`（内存紧张时可跳过）

分块读 → 每块并入查找表、算新列、丢旧列 → 最后一次性写出。

峰值内存约等于整个 `full_data` 在 pandas 里的大小。**跑不动就跳过**，回归不依赖这两列。

In [ ]:
parts = []
for i, ch in enumerate(pd.read_stata(FULL, chunksize=10_000_000), 1):
    ch = ch.drop(columns=['main_product_output', 'sales_relative_main'])
    ch = ch.merge(mainprod, on=['year', 'firm_id'], how='left')
    ch['production_relative_main'] = ch['production_value'] / ch['main_product_production']
    parts.append(ch)
    print(f'  chunk {i}')

df = pd.concat(parts, ignore_index=True)
del parts; gc.collect()

# 列序对齐 02 的输出
cols20 = ['year', 'firm_id', 'product_id', 'total_output', 'outsourcing_value', 'production_value',
          'outsourcing_percen', 'sales_percen', 'production_relative_main', 'is_main', 'main_product',
          'main_product_production', 'input_similarity', 'output_similarity', 'firm_total_output',
          'firm_total_outsource', 'n_products', 'outsourcing_intensity', 'is_intermediary', 'is_outsourcing']
df = df[cols20 + [c for c in df.columns if c not in cols20]]

print('改写后:', f'{len(df):,}', '行 x', df.shape[1], '列')
df.to_stata(FULL, write_index=False)
print('已覆盖:', FULL.name)

## Step 3　检查

`production_relative_main` 应当恒 ≤ 1，且主产品行恰好 = 1。

In [ ]:
s = df['production_relative_main']
print('分布:'); print(s.describe(percentiles=[.5, .9, .99]).to_string())
print('\n> 1 的行数:', int((s > 1 + 1e-9).sum()), '（应为 0）')
print('主产品行不等于 1 的:', int((df.loc[df['is_main'] == 1, 'production_relative_main'] - 1).abs().gt(1e-9).sum()),
      '（应为 0；主产品 production_value = 0 时为 NaN）')
print('缺失:', int(s.isna().sum()))